# Stock Sentiment Classifier (Logistic Regression baseline)

Same pipeline as `IMDB_Logisitic_Regression.ipynb`, adapted to `stock_data.csv`.

In [ ]:
import re
import time
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay, classification_report
)

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, LSTM, GRU, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

print("TensorFlow version:", tf.__version__)


## 1. Load the dataset

In [ ]:
df = pd.read_csv("stock_data.csv")
print("Shape:", df.shape)
df.head(3)


## 2. Drop duplicates

In [ ]:
df.duplicated().sum()

In [ ]:
df.drop_duplicates(inplace=True)

In [ ]:
df.duplicated().sum()

## 3. Clean the text

In [ ]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"<.*?>", " ", text)        # remove HTML tags e.g. <br />
    text = re.sub(r"[^a-z\s]", " ", text)     # keep only letters
    text = re.sub(r"\s+", " ", text).strip()  # collapse whitespace
    return text


In [ ]:
df["clean_text"] = df["Text"].apply(clean_text)

## 4. Map labels

`1` (positive) / `-1` (negative) -> `1` / `0`

In [ ]:
df["label"] = df["Sentiment"].map({1: 1, -1: 0})

## 5. Train/test split

In [ ]:
X_train_text, X_test_text, y_train, y_test = train_test_split(
    df["clean_text"], df["label"],
    test_size=0.2, random_state=42, stratify=df["label"]
)


## 6. TF-IDF vectorize

In [ ]:
tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))

In [ ]:
X_train_tfidf = tfidf.fit_transform(X_train_text)
X_test_tfidf = tfidf.transform(X_test_text)


## 7. Train logistic regression

In [ ]:
baseline = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
baseline.fit(X_train_tfidf, y_train)


## 8. Evaluate

In [ ]:
baseline_preds = baseline.predict(X_test_tfidf)
print(classification_report(y_test, baseline_preds))
accuracy_score(y_test, baseline_preds)


## 9. Save the model and vectorizer

In [ ]:
with open("stock_logreg_model.pkl", "wb") as f:
    pickle.dump(baseline, f)

with open("stock_tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(tfidf, f)
